# BEV CNN Policy — Spatial Ego-History Representation

**Motivation:** The BC MLP uses only a 6-dim state vector `[sin(yaw), cos(yaw), vx, vy, ax, ay]`
as input. This captures the current dynamic state but has no spatial context:
- No sense of *where* the ego has been in the last second
- No way to distinguish a sharp deceleration from a gradual stop
- No directional history (was the ego turning left before this moment?)

A Bird's-Eye-View (BEV) CNN replaces the scalar state with a top-down rasterized image
of the ego's trajectory history. The spatial structure lets the CNN learn patterns like
"I've been curving right for the last 10 timesteps → continue curving."

**What this notebook builds:**
1. BEV rasterizer: ego history (10 timesteps × 5 dims) → (3, 64, 64) spatial image
2. BEVPolicy: CNN encoder + state MLP + FC head → 48-dim future trajectory
3. On-the-fly BEV Dataset (memory-efficient)
4. Training loop + ADE/FDE eval vs. BC MLP baseline
5. BEVPlanner: AbstractPlanner wrapper for nuPlan closed-loop

**REF:** BEV representations in AV: Hu et al. (2022) MILE, Caesar et al. (2021) nuPlan.
Grid design follows Liang et al. (2020) LaneGCN convention (ego-centered, heading-up).


In [ ]:
# Cell 1 — Imports and config
import os, sys, sqlite3
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

sys.path.insert(0, '/Users/parvpatodia/nuplan-devkit')
sys.path.insert(0, '/Users/parvpatodia/Desktop/diffusion-policy-zoo/nuplan')

os.environ.setdefault('NUPLAN_DATA_ROOT', '/Users/parvpatodia/nuplan-devkit/data/cache')
os.environ.setdefault('NUPLAN_MAPS_ROOT', '/Users/parvpatodia/nuplan-devkit/maps')
os.environ.setdefault('NUPLAN_EXP_ROOT',  '/Users/parvpatodia/nuplan-devkit/exp')
os.environ.setdefault('NUPLAN_TUTORIAL_PATH', '/Users/parvpatodia/nuplan-devkit/tutorials')

# ── Grid config ──────────────────────────────────────────────────────────────
HISTORY_STEPS = 10     # 1.0s of ego history at nuPlan's 10Hz
FUTURE_STEPS  = 16     # 1.6s prediction horizon (same as BC MLP)
GRID_H = GRID_W = 64   # pixels
M_PER_PIX     = 0.5    # spatial resolution: 0.5m/pixel → 32m × 32m FOV
BEV_CHANNELS  = 3      # temporal occupancy, speed, heading-angle
V_MAX         = 20.0   # m/s — normalises channel 1 to [0,1]
STRIDE        = 50     # sample every 50th row → ~50K total windows from 64 DBs

# ── Paths ─────────────────────────────────────────────────────────────────────
DB_DIR    = Path('/Users/parvpatodia/nuplan-devkit/data/cache/mini')
CKPT_BC   = Path('checkpoints/bc_best.pt')       # BC MLP baseline for comparison
CKPT_BEV  = Path('checkpoints/bev_cnn.pt')       # this notebook writes this
DEVICE    = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Device:   {DEVICE}')
print(f'DB files: {len(list(DB_DIR.glob("*.db")))}')
print(f'BEV grid: {GRID_H}×{GRID_W} px at {M_PER_PIX}m/px = {GRID_H*M_PER_PIX:.0f}m × {GRID_W*M_PER_PIX:.0f}m FOV')
print(f'History:  {HISTORY_STEPS} steps = {HISTORY_STEPS*0.1:.1f}s')


In [ ]:
# Cell 2 — BEV rasterizer
#
# Converts a sequence of 10 ego poses (global frame) into a (3, 64, 64) image
# centered on the CURRENT ego pose with heading pointing toward row 0 (image top).
#
# Channel layout:
#   Ch 0 — TEMPORAL OCCUPANCY: pixel = (t+1)/T, oldest history = 0.1, current = 1.0
#           WHY: gives the CNN a temporal gradient — it can distinguish "just arrived"
#           from "been here a while". Recent positions dominate.
#   Ch 1 — SPEED MAGNITUDE: pixel = clip(speed / V_MAX, 0, 1)
#           WHY: the CNN needs to know if the ego was fast or slow at each past pose.
#           A speed=0 (stopped) looks the same as speed=15 m/s in Ch 0 alone.
#   Ch 2 — HEADING DELTA: pixel = (yaw_hist - yaw_current) / pi, clipped to [-1, 1]
#           WHY: encodes turning history. Straight driving = 0. Left turn = positive.
#           The CNN can detect patterns like "turning left for 5 consecutive steps".
#
# Coordinate convention (LaneGCN / standard BEV):
#   +X → right of ego  (larger pixel column)
#   +Y → forward of ego (smaller pixel row — image origin is top-left)

def rasterize_ego_bev(
    history_x:   np.ndarray,   # (T,) global x
    history_y:   np.ndarray,   # (T,) global y
    history_yaw: np.ndarray,   # (T,) heading [rad]
    history_vx:  np.ndarray,   # (T,) ego-frame longitudinal velocity
    history_vy:  np.ndarray,   # (T,) ego-frame lateral velocity
) -> np.ndarray:
    """
    Args:
        history_*: arrays of length HISTORY_STEPS, index 0 = oldest, -1 = current
    Returns:
        img: (BEV_CHANNELS, GRID_H, GRID_W) float32 image
    """
    T      = len(history_x)
    img    = np.zeros((BEV_CHANNELS, GRID_H, GRID_W), dtype=np.float32)

    # Ego reference pose (current = last element)
    cx, cy, c_yaw = history_x[-1], history_y[-1], history_yaw[-1]
    # Rotation: global → ego frame  (rotate by -c_yaw)
    cos_h = np.cos(-c_yaw)
    sin_h = np.sin(-c_yaw)

    for t in range(T):
        # Transform historical pose to ego-relative frame
        dx_w  = history_x[t]   - cx
        dy_w  = history_y[t]   - cy
        dx_e  =  cos_h * dx_w - sin_h * dy_w    # right of ego
        dy_e  =  sin_h * dx_w + cos_h * dy_w    # forward of ego

        # Pixel coordinates  (ego at grid center; forward = up in image)
        px = int(GRID_W / 2 + dx_e / M_PER_PIX)
        py = int(GRID_H / 2 - dy_e / M_PER_PIX)   # flip y: forward = smaller row

        if not (0 <= px < GRID_W and 0 <= py < GRID_H):
            continue   # out of FOV — skip (common for fast-moving logs)

        temporal_w = (t + 1) / T          # 0.1 … 1.0 (oldest → newest)

        # Ch 0: temporal occupancy
        img[0, py, px] = max(img[0, py, px], temporal_w)

        # Ch 1: speed magnitude (longitudinal + lateral, normalised)
        speed = np.sqrt(history_vx[t] ** 2 + history_vy[t] ** 2)
        img[1, py, px] = max(img[1, py, px], min(speed / V_MAX, 1.0))

        # Ch 2: heading delta (how much the ego has turned relative to now)
        d_yaw = history_yaw[t] - c_yaw
        d_yaw = (d_yaw + np.pi) % (2 * np.pi) - np.pi   # wrap to [-pi, pi]
        img[2, py, px] = d_yaw / np.pi                   # normalise to [-1, 1]

    return img


# ── Sanity check: rasterize a straight-line trajectory ───────────────────────
T_test = HISTORY_STEPS
x_test   = np.linspace(0, 5, T_test)    # 5m forward over 1s at 5 m/s
y_test   = np.zeros(T_test)
yaw_test = np.zeros(T_test)
vx_test  = np.full(T_test, 5.0)
vy_test  = np.zeros(T_test)
bev_test = rasterize_ego_bev(x_test, y_test, yaw_test, vx_test, vy_test)

print(f'BEV shape: {bev_test.shape}')
print(f'Ch 0 (occupancy): {bev_test[0].sum():.1f} active pixels  (expect {T_test})')
print(f'Ch 1 (speed):     max={bev_test[1].max():.3f} (expect {5.0/V_MAX:.3f})')
print(f'Ch 2 (heading):   max={bev_test[2].max():.3f}, min={bev_test[2].min():.3f} (expect 0.0 — straight)')
print('Rasterizer OK.')


In [ ]:
# Cell 3 — BEV Dataset
#
# Design choice: rasterize ON-THE-FLY in __getitem__ (not pre-stored).
# WHY: storing 50K × (3×64×64) × float32 = 2.4 GB RAM. Rasterizing 10 poses
#      per sample takes ~0.1ms on CPU. With 4 DataLoader workers, the bottleneck
#      remains the GPU forward pass, not rasterization. No memory pressure.
#
# Pre-extracted from DB: only the scalar arrays (x, y, yaw, vx, vy, ax, ay).
# These are O(KB) per window and fit easily in a numpy array.

def quat_to_yaw(qw, qx, qy, qz):
    return np.arctan2(2.0 * (qw * qz + qx * qy), 1.0 - 2.0 * (qy**2 + qz**2))

def load_db_array(db_path: str) -> np.ndarray:
    """
    Load ego_pose table as (N, 7) array: [x, y, yaw, vx, vy, ax, ay].
    Returns None if fewer than (HISTORY_STEPS + FUTURE_STEPS + 1) rows.
    """
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        'SELECT x, y, qw, qx, qy, qz, vx, vy, acceleration_x, acceleration_y '
        'FROM ego_pose ORDER BY timestamp'
    ).fetchall()
    conn.close()
    if len(rows) < HISTORY_STEPS + FUTURE_STEPS + 1:
        return None
    arr = np.array(rows, dtype=np.float64)
    yaw = quat_to_yaw(arr[:,2], arr[:,3], arr[:,4], arr[:,5])
    # Return [x, y, yaw, vx, vy, ax, ay]
    return np.column_stack([arr[:,0], arr[:,1], yaw,
                            arr[:,6], arr[:,7], arr[:,8], arr[:,9]]).astype(np.float32)


class BEVDataset(Dataset):
    """
    On-the-fly BEV rasterization from pre-loaded scalar arrays.

    Each sample:
      bev:   (3, 64, 64)  — rasterized ego history (HISTORY_STEPS frames)
      state: (6,)         — [sin(yaw), cos(yaw), vx, vy, ax, ay] at current step
      tgt:   (48,)        — (dx, dy, d_yaw) × 16 future steps, ego-relative

    WHY separate 'state' alongside 'bev':
      The BEV image captures spatial/temporal structure but loses precise scalar
      values (vx=13.7 vs 14.2 m/s look identical after rasterisation at 0.5m/px).
      The 6-dim state provides exact scalars for the MLP branch.
    """

    def __init__(self, db_files, stride=STRIDE, bev_norm_mean=None, bev_norm_std=None):
        self.segments  = []   # list of (arr, start_idx) — one per valid window
        self._collect(db_files, stride)

    def _collect(self, db_files, stride):
        for db_path in db_files:
            arr = load_db_array(str(db_path))
            if arr is None:
                continue
            N = len(arr)
            for i in range(HISTORY_STEPS, N - FUTURE_STEPS, stride):
                self.segments.append((arr, i))
        print(f'BEVDataset: {len(self.segments):,} windows from {len(db_files)} DB files')

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        arr, i = self.segments[idx]

        # ── BEV rasterization ─────────────────────────────────────────────
        hist_slice = arr[i - HISTORY_STEPS : i]   # (HISTORY_STEPS, 7)
        bev = rasterize_ego_bev(
            history_x   = hist_slice[:, 0],
            history_y   = hist_slice[:, 1],
            history_yaw = hist_slice[:, 2],
            history_vx  = hist_slice[:, 3],
            history_vy  = hist_slice[:, 4],
        )

        # ── Current ego state (6-dim) ─────────────────────────────────────
        yaw = arr[i, 2]
        state = np.array([
            np.sin(yaw), np.cos(yaw),
            arr[i, 3], arr[i, 4],   # vx, vy
            arr[i, 5], arr[i, 6],   # ax, ay
        ], dtype=np.float32)

        # ── Future trajectory (48-dim, ego-relative) ──────────────────────
        cx, cy, cyaw = arr[i, 0], arr[i, 1], arr[i, 2]
        cos_h = np.cos(-cyaw)
        sin_h = np.sin(-cyaw)
        tgt = np.zeros(FUTURE_STEPS * 3, dtype=np.float32)
        for j in range(FUTURE_STEPS):
            fi = i + j + 1
            dx_w = arr[fi, 0] - cx
            dy_w = arr[fi, 1] - cy
            dx_e =  cos_h * dx_w - sin_h * dy_w
            dy_e =  sin_h * dx_w + cos_h * dy_w
            d_yaw = arr[fi, 2] - cyaw
            d_yaw = (d_yaw + np.pi) % (2 * np.pi) - np.pi
            tgt[j * 3]     = dx_e
            tgt[j * 3 + 1] = dy_e
            tgt[j * 3 + 2] = d_yaw

        return (
            torch.from_numpy(bev),
            torch.from_numpy(state),
            torch.from_numpy(tgt),
        )


# ── Build train / val splits ──────────────────────────────────────────────────
all_dbs   = sorted(DB_DIR.glob('*.db'))
np.random.seed(42)
db_perm   = np.random.permutation(len(all_dbs))
n_tr_dbs  = int(0.8 * len(all_dbs))   # 80% logs for training
tr_dbs    = [all_dbs[i] for i in db_perm[:n_tr_dbs]]
va_dbs    = [all_dbs[i] for i in db_perm[n_tr_dbs:]]

print('Building datasets ...')
train_ds = BEVDataset(tr_dbs, stride=STRIDE)
val_ds   = BEVDataset(va_dbs, stride=STRIDE)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')
print(f'Sample shapes — BEV: {train_ds[0][0].shape}, state: {train_ds[0][1].shape}, tgt: {train_ds[0][2].shape}')


In [ ]:
# Cell 4 — Compute normalization statistics
#
# WHY normalise state + target separately from BC MLP:
#   BEVDataset uses a different stride (50 vs 10) and a different DB split.
#   Reusing BC stats would introduce a subtle distribution mismatch.
#   Compute fresh stats from the BEV training set.
#
# BEV images are NOT normalised per-pixel because:
#   Ch 0: already in [0, 1] by construction (temporal weight)
#   Ch 1: already in [0, 1] by construction (speed / V_MAX)
#   Ch 2: already in [-1, 1] by construction (heading delta / pi)
#   Applying per-pixel batch-norm would be correct but adds complexity
#   with minimal gain for a first baseline.

from tqdm import tqdm

# Collect state + target arrays from train set (no BEV needed for stats)
# WHY: sample every 10th item for speed — stats are stable with 5K samples
sample_idx = list(range(0, len(train_ds), max(1, len(train_ds) // 5000)))
states, tgts = [], []
for i in tqdm(sample_idx, desc='Computing stats', ncols=80):
    _, s, t = train_ds[i]
    states.append(s.numpy())
    tgts.append(t.numpy())

S_arr = np.array(states, dtype=np.float32)
T_arr = np.array(tgts,   dtype=np.float32)

S_mean = S_arr.mean(0);  S_std = S_arr.std(0) + 1e-8
T_mean = T_arr.mean(0);  T_std = T_arr.std(0) + 1e-8

print(f'State mean: {S_mean.round(3)}')
print(f'State std:  {S_std.round(3)}')
print(f'Tgt mean range: [{T_mean.min():.3f}, {T_mean.max():.3f}]')
print(f'Tgt std  range: [{T_std.min():.3f}, {T_std.max():.3f}]')

# Convert to tensors for use in training loop
S_mean_t = torch.tensor(S_mean, dtype=torch.float32)
S_std_t  = torch.tensor(S_std,  dtype=torch.float32)
T_mean_t = torch.tensor(T_mean, dtype=torch.float32)
T_std_t  = torch.tensor(T_std,  dtype=torch.float32)


In [ ]:
# Cell 5 — BEVPolicy architecture
#
# Two-branch architecture:
#
#   Branch A — CNN encoder:
#     Input: (3, 64, 64) BEV image
#     3 conv blocks: (32→64→128 channels), each block = Conv + ReLU + Conv + ReLU + MaxPool2d
#     AdaptiveAvgPool(1) → Flatten → 128-dim feature vector
#     WHY global avg pool (not flatten): invariant to minor rasterization jitter,
#     smaller head, regularised implicitly.
#
#   Branch B — State MLP:
#     Input: (6,) normalised ego state
#     2-layer MLP: 6 → 64 → 64
#     WHY keep separate from CNN: precise scalar values (vx=13.7 m/s) are
#     lost after rasterization; state branch recovers this signal cleanly.
#
#   Head:
#     Concat(128 + 64) → Linear(192, 256) → ReLU → Linear(256, 48)
#     Output: 48-dim normalised trajectory
#
# Parameters: ~370K  (BC MLP: ~260K — 1.4x larger)
# REF: architecture loosely follows the BEV encoder in VectorNet ablation
#      (Gao et al. 2020) with reduced depth for our single-log mini dataset.

class BEVPolicy(nn.Module):
    """
    BEV CNN + ego state → trajectory.
    Input:  bev   (B, 3, 64, 64)
            state (B, 6)  -- normalised
    Output: traj  (B, 48) -- normalised (dx, dy, d_yaw) × 16
    """

    def __init__(
        self,
        bev_ch:    int = BEV_CHANNELS,
        state_dim: int = 6,
        out_dim:   int = FUTURE_STEPS * 3,   # 48
    ):
        super().__init__()

        # ── CNN encoder ──────────────────────────────────────────────────
        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.encoder = nn.Sequential(
            conv_block(bev_ch, 32),    # (3,64,64) → (32,32,32)
            conv_block(32,     64),    # (32,32,32) → (64,16,16)
            conv_block(64,     128),   # (64,16,16) → (128,8,8)
            nn.AdaptiveAvgPool2d(1),   # (128,8,8) → (128,1,1)
            nn.Flatten(),              # → (128,)
        )

        # ── State branch ─────────────────────────────────────────────────
        self.state_enc = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(inplace=True),
            nn.Linear(64, 64),        nn.ReLU(inplace=True),
        )

        # ── Prediction head ───────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(128 + 64, 256), nn.ReLU(inplace=True),
            nn.Linear(256, out_dim),
        )

    def forward(self, bev: torch.Tensor, state: torch.Tensor) -> torch.Tensor:
        feat_bev   = self.encoder(bev)            # (B, 128)
        feat_state = self.state_enc(state)        # (B, 64)
        return self.head(torch.cat([feat_bev, feat_state], dim=-1))  # (B, 48)


# ── Parameter count ───────────────────────────────────────────────────────────
model_bev = BEVPolicy().to(DEVICE)
n_params  = sum(p.numel() for p in model_bev.parameters())
print(f'BEVPolicy parameters: {n_params:,}')

# BC MLP for comparison
from planners import BCPolicy
bc_params = sum(p.numel() for p in BCPolicy().parameters())
print(f'BCPolicy  parameters: {bc_params:,}')
print(f'BEV/BC ratio: {n_params/bc_params:.1f}x')

# Forward pass sanity check
bev_t   = torch.zeros(4, BEV_CHANNELS, GRID_H, GRID_W).to(DEVICE)
state_t = torch.zeros(4, 6).to(DEVICE)
out     = model_bev(bev_t, state_t)
print(f'Output shape: {out.shape}  (expect (4, 48))')


In [ ]:
# Cell 6 — Training loop
#
# Identical structure to bc_pipeline.ipynb training, adapted for dual-input model.
# Differences:
#   - DataLoader with num_workers=2 (rasterization is CPU-bound, parallelise)
#   - Normalise state + target inline using tensors (no pre-normalised arrays)
#   - Gradient clip (max_norm=1.0): BEV encoder gradients can spike early in training
#   - 30 epochs (same as DAgger iter 2 — slightly more capacity to absorb BEV signal)

EPOCHS     = 30
BATCH_SIZE = 256   # smaller than BC (256 vs 512) due to BEV memory per sample
LR         = 1e-3
WORKERS    = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True)

optimizer  = torch.optim.Adam(model_bev.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5, verbose=True)

# Move norm stats to device
_S_mean = S_mean_t.to(DEVICE)
_S_std  = S_std_t.to(DEVICE)
_T_mean = T_mean_t.to(DEVICE)
_T_std  = T_std_t.to(DEVICE)

best_val    = float('inf')
train_losses, val_losses = [], []

print(f'Training BEVPolicy for {EPOCHS} epochs  (batches/epoch ≈ {len(train_loader)})')
for epoch in range(EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────
    model_bev.train()
    ep_loss = 0.0
    for bev_b, state_b, tgt_b in train_loader:
        bev_b   = bev_b.to(DEVICE)
        state_b = ((state_b.to(DEVICE) - _S_mean) / _S_std)
        tgt_b   = ((tgt_b.to(DEVICE)  - _T_mean) / _T_std)
        pred    = model_bev(bev_b, state_b)
        loss    = nn.functional.mse_loss(pred, tgt_b)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bev.parameters(), max_norm=1.0)
        optimizer.step()
        ep_loss += loss.item() * len(bev_b)
    ep_loss /= len(train_ds)

    # ── Validate ──────────────────────────────────────────────────────────
    model_bev.eval()
    val_loss = 0.0
    with torch.no_grad():
        for bev_b, state_b, tgt_b in val_loader:
            bev_b   = bev_b.to(DEVICE)
            state_b = ((state_b.to(DEVICE) - _S_mean) / _S_std)
            tgt_b   = ((tgt_b.to(DEVICE)  - _T_mean) / _T_std)
            pred    = model_bev(bev_b, state_b)
            val_loss += nn.functional.mse_loss(pred, tgt_b).item() * len(bev_b)
    val_loss /= len(val_ds)

    scheduler.step(val_loss)
    train_losses.append(ep_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'model':   model_bev.state_dict(),
            'S_mean':  S_mean, 'S_std': S_std,
            'T_mean':  T_mean, 'T_std': T_std,
        }, CKPT_BEV)

    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}: train={ep_loss:.5f}  val={val_loss:.5f}  best={best_val:.5f}')

print(f'\nBEVPolicy saved to {CKPT_BEV}')
print(f'Best val loss: {best_val:.5f}')


In [ ]:
# Cell 7 — Open-loop ADE / FDE evaluation
#
# Compare BEVPolicy vs BC MLP on the same 2,000 held-out validation windows.
# WHY same eval as bc_pipeline.ipynb:
#   ADE/FDE on the original 80/10/10 val split is the standard metric.
#   Using the same sample set makes the comparison fair — both policies
#   predict from ground-truth expert states (no closed-loop drift here).
#
# Expected result:
#   BEV ADE should be <= BC ADE (0.058m). The BEV gets more input signal
#   (spatial history vs scalar state), so it should have at least equal or
#   better open-loop accuracy. If BEV ADE > BC ADE, the CNN hasn't learned
#   the spatial patterns yet — try more epochs or smaller stride.

from planners import BCPolicy as BCPolicyClass

# Load BEV model
ckpt_bev = torch.load(CKPT_BEV, map_location='cpu', weights_only=False)
model_bev_eval = BEVPolicy().eval()
model_bev_eval.load_state_dict(ckpt_bev['model'])

# Load BC baseline
ckpt_bc  = torch.load(CKPT_BC, map_location='cpu', weights_only=False)
model_bc  = BCPolicyClass().eval()
model_bc.load_state_dict(ckpt_bc['model'])

# Build eval dataset from val DB files (same val_dbs split from Cell 3)
# WHY stride=STRIDE not 10: eval on the same sampling grid as training
#     to avoid measuring performance on unseen interpolated states.
eval_ds  = BEVDataset(va_dbs, stride=STRIDE)

rng = np.random.default_rng(42)
eval_idx = rng.choice(len(eval_ds), min(2000, len(eval_ds)), replace=False)

ade_bev, fde_bev = [], []
ade_bc,  fde_bc  = [], []

_s_mean_np = S_mean;  _s_std_np = S_std
_t_mean_np = T_mean;  _t_std_np = T_std

with torch.no_grad():
    for idx in eval_idx:
        bev_i, state_i, tgt_i = eval_ds[idx]
        gt = tgt_i.numpy().reshape(FUTURE_STEPS, 3)   # (16, 3) ground truth

        # ── BEV model ──────────────────────────────────────────────────
        bev_in   = bev_i.unsqueeze(0)
        state_in = ((state_i - torch.tensor(_s_mean_np)) /
                     torch.tensor(_s_std_np)).unsqueeze(0)
        pred_bev_norm = model_bev_eval(bev_in, state_in).squeeze(0).numpy()
        pred_bev = (pred_bev_norm * _t_std_np + _t_mean_np).reshape(FUTURE_STEPS, 3)

        # ── BC MLP (uses its own normalization stats) ──────────────────
        state_bc_norm = ((state_i.numpy() - ckpt_bc['X_mean']) / ckpt_bc['X_std'])
        x_bc = torch.tensor(state_bc_norm, dtype=torch.float32).unsqueeze(0)
        pred_bc_norm  = model_bc(x_bc).squeeze(0).numpy()
        pred_bc = (pred_bc_norm * ckpt_bc['Y_std'] + ckpt_bc['Y_mean']).reshape(FUTURE_STEPS, 3)

        # ── ADE / FDE ──────────────────────────────────────────────────
        def ade_fde(pred, gt):
            d = np.sqrt(np.sum((pred[:, :2] - gt[:, :2])**2, axis=1))
            return d.mean(), d[-1]

        a_bev, f_bev = ade_fde(pred_bev, gt)
        a_bc,  f_bc  = ade_fde(pred_bc,  gt)
        ade_bev.append(a_bev);  fde_bev.append(f_bev)
        ade_bc.append(a_bc);    fde_bc.append(f_bc)

print(f"{'Policy':<16} {'ADE (m)':>10} {'FDE (m)':>10}")
print('-' * 38)
print(f"{'BEV CNN':<16} {np.mean(ade_bev):>10.3f} {np.mean(fde_bev):>10.3f}")
print(f"{'BC MLP':<16} {np.mean(ade_bc):>10.3f} {np.mean(fde_bc):>10.3f}")
print(f"\nBEV vs BC ADE: {(np.mean(ade_bev)/np.mean(ade_bc)-1)*100:+.1f}%")


In [ ]:
# Cell 8 — BEVPlanner: AbstractPlanner wrapper for nuPlan closed-loop
#
# Wraps BEVPolicy as a drop-in AbstractPlanner.
# The key implementation detail: maintaining an ego-state buffer.
# PlannerInput.history gives us the last N ego states + timestamps.
# We extract the last HISTORY_STEPS from that buffer to build the BEV image.
#
# If fewer than HISTORY_STEPS states are in the buffer (start of scenario),
# we pad with the oldest available state (zero-order hold).
# WHY zero-order hold: repeating the oldest known pose is a neutral assumption.
# Alternatives (zero-fill, linear extrapolation) introduce artificial motion.

import sys
sys.path.insert(0, '/Users/parvpatodia/nuplan-devkit')
sys.path.insert(0, '/Users/parvpatodia/Desktop/diffusion-policy-zoo/nuplan')

import numpy as np
import torch
from nuplan.common.actor_state.ego_state import EgoState
from nuplan.common.actor_state.state_representation import StateSE2, StateVector2D, TimePoint
from nuplan.planning.simulation.observation.observation_type import DetectionsTracks
from nuplan.planning.simulation.planner.abstract_planner import (
    AbstractPlanner, PlannerInitialization, PlannerInput,
)
from nuplan.planning.simulation.trajectory.interpolated_trajectory import InterpolatedTrajectory

DT = 0.1   # nuPlan 10Hz


class BEVPlanner(AbstractPlanner):
    """
    AbstractPlanner wrapper for BEVPolicy.
    Maintains a rolling HISTORY_STEPS buffer of ego states.
    Rasterizes BEV image at each planning step and runs one forward pass.
    """

    def __init__(self, ckpt_path: str):
        self._ckpt_path = ckpt_path
        self._device    = torch.device('cpu')   # WHY: CPU for macOS MPS sim stability
        self._model     = None
        self._S_mean = self._S_std = None
        self._T_mean = self._T_std = None
        self._history   = []   # rolling buffer of (x, y, yaw, vx, vy) tuples

    def name(self) -> str:
        return 'BEVPlanner'

    def observation_type(self):
        return DetectionsTracks

    def initialize(self, initialization: PlannerInitialization) -> None:
        ckpt = torch.load(self._ckpt_path, map_location=self._device, weights_only=False)
        self._model = BEVPolicy().to(self._device)
        self._model.load_state_dict(ckpt['model'])
        self._model.eval()
        self._S_mean = torch.tensor(ckpt['S_mean'], dtype=torch.float32)
        self._S_std  = torch.tensor(ckpt['S_std'],  dtype=torch.float32)
        self._T_mean = ckpt['T_mean']
        self._T_std  = ckpt['T_std']
        self._history = []

    def _ego_to_tuple(self, ego: EgoState):
        dcs = ego.dynamic_car_state
        return (
            ego.rear_axle.x,
            ego.rear_axle.y,
            ego.rear_axle.heading,
            dcs.rear_axle_velocity_2d.x,
            dcs.rear_axle_velocity_2d.y,
        )

    def _build_bev(self) -> torch.Tensor:
        """Pad or slice history to HISTORY_STEPS, rasterise, return (1,3,H,W)."""
        buf = self._history
        if len(buf) < HISTORY_STEPS:
            # Zero-order hold: pad with oldest entry
            pad = [buf[0]] * (HISTORY_STEPS - len(buf))
            buf = pad + list(buf)
        else:
            buf = list(buf[-HISTORY_STEPS:])

        hist = np.array(buf, dtype=np.float32)   # (10, 5)
        bev  = rasterize_ego_bev(
            history_x   = hist[:, 0],
            history_y   = hist[:, 1],
            history_yaw = hist[:, 2],
            history_vx  = hist[:, 3],
            history_vy  = hist[:, 4],
        )
        return torch.from_numpy(bev).unsqueeze(0)   # (1, 3, 64, 64)

    def compute_planner_trajectory(self, current_input: PlannerInput) -> InterpolatedTrajectory:
        ego = current_input.history.current_state[0]

        # Update rolling history
        self._history.append(self._ego_to_tuple(ego))

        # Build inputs
        bev_t = self._build_bev().to(self._device)
        yaw   = ego.rear_axle.heading
        dcs   = ego.dynamic_car_state
        state_np = np.array([
            np.sin(yaw), np.cos(yaw),
            dcs.rear_axle_velocity_2d.x,
            dcs.rear_axle_velocity_2d.y,
            dcs.rear_axle_acceleration_2d.x,
            dcs.rear_axle_acceleration_2d.y,
        ], dtype=np.float32)
        state_t = torch.tensor(
            (state_np - self._S_mean.numpy()) / self._S_std.numpy(),
            dtype=torch.float32
        ).unsqueeze(0).to(self._device)

        # Forward pass
        with torch.no_grad():
            pred_norm = self._model(bev_t, state_t).squeeze(0).numpy()
        pred = (pred_norm * self._T_std + self._T_mean).reshape(FUTURE_STEPS, 3)

        # Build InterpolatedTrajectory (same as BCPlanner)
        cx, cy    = ego.rear_axle.x, ego.rear_axle.y
        cos_h, sin_h = np.cos(yaw), np.sin(yaw)
        vx = dcs.rear_axle_velocity_2d.x
        vy = dcs.rear_axle_velocity_2d.y
        ax = dcs.rear_axle_acceleration_2d.x
        ay = dcs.rear_axle_acceleration_2d.y
        t0 = ego.time_point.time_us

        states = [ego]
        for j, (dx_e, dy_e, d_yaw) in enumerate(pred):
            wx    = cx + cos_h * dx_e - sin_h * dy_e
            wy    = cy + sin_h * dx_e + cos_h * dy_e
            w_yaw = yaw + d_yaw
            states.append(EgoState.build_from_rear_axle(
                rear_axle_pose=StateSE2(wx, wy, w_yaw),
                rear_axle_velocity_2d=StateVector2D(vx, vy),
                rear_axle_acceleration_2d=StateVector2D(ax, ay),
                tire_steering_angle=0.0,
                time_point=TimePoint(t0 + int((j + 1) * DT * 1e6)),
                vehicle_parameters=ego.car_footprint.vehicle_parameters,
            ))
        return InterpolatedTrajectory(states)


print('BEVPlanner class defined.')
print('To run closed-loop eval, add BEVPlanner(str(CKPT_BEV)) to closed_loop_eval.py.')


## Architecture notes

### What the BEV adds over BC MLP

| Feature | BC MLP (6-dim) | BEV CNN (3×64×64 + 6-dim) |
|---|---|---|
| Current velocity | Yes (vx, vy) | Yes (state branch) |
| Current acceleration | Yes (ax, ay) | Yes (state branch) |
| Trajectory history (1s) | No | Yes — channel 0 |
| Speed history | No | Yes — channel 1 |
| Turning history | No | Yes — channel 2 |
| Spatial context | No | Yes — relative position of past poses |

### What the BEV does NOT add (yet)

- Road geometry (lanes, centerlines, stop lines) — requires `map_api.get_map_objects()`
- Other agent positions and velocities — requires DetectionsTracks rasterization
- Both of the above are the standard BEV representation in production AV systems
  (see nuScenes-devkit, Waymo Open Dataset, UniAD)

### Expected performance

- Open-loop ADE: BEV CNN should match or slightly beat BC MLP (0.058m)
  because it has strictly more input signal. If BEV ADE > BC ADE, the CNN
  is not fully utilising the spatial history.
- Closed-loop L2: unclear without running the eval. BEV CNN is still a
  pure imitation learner — it will suffer from the same covariate shift as BC MLP.
  The BEV representation alone does not fix covariate shift; that requires DAgger
  or a world model.

### Ablation ideas
1. **Map layer**: add a 4th BEV channel from `map_api` road polygon rasterization
2. **Agent layer**: add a 5th/6th channel for DetectionsTracks bounding boxes
3. **Larger grid**: 128×128 at 0.25m/px → 32m FOV at higher resolution
4. **Temporal encoding**: replace Gaussian blobs with sinusoidal position encoding
5. **DAgger + BEV**: apply DAgger to BEVPlanner — same pipeline as `dagger.ipynb`

### Next: MILE world model (Phase 3)
Latent-space world model trained to:
1. Encode current BEV → latent state z_t
2. Predict z_{t+1} from (z_t, action_t)
3. Decode trajectory from imagined z_{t+1:t+16}
REF: Hu et al. (2022) MILE. arXiv:2209.14430
